In [ ]:
# Vizualizáció: minta képek annotációkkal és statisztikák

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 3 minta kép bboxokkal
sample_ids = random.sample(train_images[:500], 3)
for i, img_id in enumerate(sample_ids):
    img_path = images_dir / f"{img_id}.jpg"
    img = Image.open(img_path)
    xml_path = annotations_dir / f"{img_id}.xml"
    objects, _ = parse_voc_xml(xml_path)

    axes[0, i].imshow(img)
    for cls, bbox in objects:
        rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                                   linewidth=2, edgecolor='red', facecolor='none')
        axes[0, i].add_patch(rect)
        axes[0, i].text(bbox[0], bbox[1]-5, cls, color='red', fontsize=8)
    axes[0, i].set_title(f"{img_id} ({len(objects)} objektum)")
    axes[0, i].axis('off')

# Osztályeloszlás hisztogram
axes[1, 0].bar(range(len(VOC_CLASSES)), [class_counts[c] for c in VOC_CLASSES])
axes[1, 0].set_xticks(range(len(VOC_CLASSES)))
axes[1, 0].set_xticklabels(VOC_CLASSES, rotation=90, fontsize=7)
axes[1, 0].set_title("Osztályeloszlás")

# Bbox terület hisztogram
axes[1, 1].hist(bbox_areas, bins=50, edgecolor='black')
axes[1, 1].set_title("Bbox területek eloszlása")
axes[1, 1].set_xlabel("Terület (px²)")

# Bbox arány hisztogram
axes[1, 2].hist(bbox_ratios, bins=50, edgecolor='black')
axes[1, 2].set_title("Bbox szélesség/magasság arány")
axes[1, 2].set_xlabel("w/h arány")

plt.tight_layout()
plt.savefig('dataset_exploration.png', dpi=100)
plt.show()
print("Adathalmaz feltérképezés kész.")

In [ ]:
# Osztályok listája
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

def parse_voc_xml(xml_path):
    """VOC XML annotáció parse-olása. Visszaadja: [(osztály, [xmin,ymin,xmax,ymax]), ...]"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)
    for obj in root.findall('object'):
        name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        objects.append((name, [xmin, ymin, xmax, ymax]))
    return objects, (width, height)

# Osztályeloszlás vizsgálata
class_counts = defaultdict(int)
bbox_areas = []
bbox_ratios = []
image_sizes = []

for img_id in train_images[:500]:  # első 500 kép a gyors elemzéshez
    xml_path = annotations_dir / f"{img_id}.xml"
    if xml_path.exists():
        objects, (w, h) = parse_voc_xml(xml_path)
        image_sizes.append((w, h))
        for cls, bbox in objects:
            class_counts[cls] += 1
            area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
            ratio = (bbox[2] - bbox[0]) / max(bbox[3] - bbox[1], 1)
            bbox_areas.append(area)
            bbox_ratios.append(ratio)

print("=== Osztályeloszlás (első 500 train kép) ===")
for cls in VOC_CLASSES:
    print(f"  {cls}: {class_counts[cls]}")

print(f"\nBbox terület: min={min(bbox_areas):.0f}, max={max(bbox_areas):.0f}, mean={np.mean(bbox_areas):.0f}")
print(f"Bbox arány (w/h): mean={np.mean(bbox_ratios):.2f}")
print(f"Képméret: min={min(image_sizes)}, max={max(image_sizes)}")

In [ ]:
# Pascal VOC 2012 letöltése
path = kagglehub.dataset_download("gopalbhattrai/pascal-voc-2012-dataset")
print(f"Adathalmaz elérési út: {path}")

# Mappaszerkezet feltárása
voc_root = Path(path) / "VOC2012"
annotations_dir = voc_root / "Annotations"
images_dir = voc_root / "JPEGImages"
imgset_dir = voc_root / "ImageSets" / "Main"

print(f"Annotations: {annotations_dir.exists()}")
print(f"Images: {images_dir.exists()}")

# Képek listája train/val splitből
train_images = []
for class_file in sorted(imgset_dir.glob("*_train.txt")):
    with open(class_file) as f:
        train_images.extend([line.strip().split()[0] for line in f if line.strip()])
train_images = list(set(train_images))

val_images = []
for class_file in sorted(imgset_dir.glob("*_val.txt")):
    with open(class_file) as f:
        val_images.extend([line.strip().split()[0] for line in f if line.strip()])
val_images = list(set(val_images))

print(f"Train képek: {len(train_images)}")
print(f"Val képek: {len(val_images)}")

# RCNN Object Detection - Pascal VOC 2012

Klasszikus RCNN (Girshick et al. 2014) implementáció PyTorch és scikit-learn használatával.
ResNet50 backbone, Selective Search régió javaslatok, SVM osztályozás.

## 0. Környezet beállítása

In [ ]:
# GPU ellenőrzés
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA elérhető: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Függőségek telepítése
!pip install -q kagglehub opencv-python scikit-learn matplotlib pillow torch torchvision

import kagglehub
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.svm import LinearSVC
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import xml.etree.ElementTree as ET
import os
import random
import pickle
from collections import defaultdict
from pathlib import Path

print("Minden függőség betöltve.")

## 1. Adathalmaz feltérképezése

Pascal VOC 2012 letöltése kagglehub segítségével, osztályeloszlás, képméretek és bbox statisztikák vizsgálata.